# Initialization

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

# Check the data integration

In [0]:
TABLE_CONFIG = [
    "bronze.erp_cust_az12",
    "bronze.crm_cust_info",
    "bronze.crm_sales_details",
    "bronze.erp_loc_a101"
]

for table in TABLE_CONFIG:
    spark.read.table(table).limit(50).display()

# Read the table

In [0]:
df = spark.read.table("bronze.erp_cust_az12")

# Trimming all Stringtype column

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

# Clean the CID column

In [0]:
df = df.withColumn("CID",
                   when(col("CID").startswith("NAS"), substring(col("CID"), 4, length(col("CID")) - 3))
                   .otherwise(col("CID")))

# Checking the correctness of date data

In [0]:
df.select("*").filter(
    (col("BDATE").isNull()) |
    (col("BDATE") > ("2020-01-01"))|
    (col("BDATE") < ("1926-02-10"))
    ).display()

we have tons of unlogic data in BDATE column

# Clean the BDATE column

In [0]:
df = df.withColumn("BDATE",
    when(col("BDATE") > ("2020-01-01"), None)
    .when(col("BDATE") < ("1926-02-10"), None)
    .otherwise(col("BDATE")))

# Check the data standardization of GEN column

In [0]:
df.select("GEN").distinct().display()

# Clean the GEN column

In [0]:
df = df.withColumn("GEN", when((col("GEN") == "") | (col("GEN").isNull()), "N/A")
              .when(col("GEN") == "M", "Male")
              .when(col("GEN") == "F", "Female")
              .otherwise(col("GEN"))
              )

# Sanity Check

In [0]:
df.display()